# 05 — Frequent Pattern Mining using FP-Growth Algorithm

**Objective:** Implement the Frequent Pattern Growth (FP-Growth) algorithm using `mlxtend`, examine its 6 core algorithmic steps, benchmark its performance against Apriori across multiple minimum support thresholds, and demonstrate how FP-Growth overcomes Apriori's computational bottlenecks.

## 1. How FP-Growth Works (The 6 Algorithmic Steps)

Unlike Apriori which relies on costly candidate generation and multiple database scans, FP-Growth uses a compact **FP-Tree (Frequent Pattern Tree)** data structure:

1. **Scan the dataset (Pass 1)**: Count the support of each individual item.
2. **Calculate frequency of each item**: Filter out infrequent items that fall below the minimum support threshold.
3. **Remove items below minimum support**: Discard non-frequent items to prune the search space.
4. **Sort items by frequency**: Order items within each transaction in descending frequency order (highest support first).
5. **Construct an FP-Tree (Pass 2)**: Read each transaction, insert items as a path in the tree, sharing common prefix nodes with incremented counts.
6. **Mine frequent patterns from the tree**: Recursively construct conditional pattern bases and conditional FP-Trees for each item suffix to mine all frequent itemsets without candidate generation.

---
## 2. Import Libraries

In [ ]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# MLxtend modules
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, apriori

sns.set_style('whitegrid')

---
## 3. Load Dataset & One-Hot Encode Transactions

In [ ]:
df = pd.read_csv('../dataset/processed/cleaned_data.csv')
df['Description'] = df['Description'].str.strip()

# Group items per transaction
transactions = df.groupby('InvoiceNo')['Description'].apply(list).tolist()

# One-hot encode
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print(f"Encoded Matrix Shape: {df_encoded.shape[0]:,} transactions × {df_encoded.shape[1]:,} items")

---
## 4. Run FP-Growth Across Different Support Values

We test FP-Growth on the exact same support thresholds (`0.05`, `0.02`, `0.01`, `0.005`) used for Apriori.

In [ ]:
support_thresholds = [0.05, 0.02, 0.01, 0.005]
fpgrowth_results = []
fpgrowth_itemsets_dict = {}

for sup in support_thresholds:
    t_start = time.time()
    f_itemsets = fpgrowth(df_encoded, min_support=sup, use_colnames=True)
    elapsed = time.time() - t_start
    
    f_itemsets['length'] = f_itemsets['itemsets'].apply(lambda x: len(x))
    fpgrowth_itemsets_dict[sup] = f_itemsets
    
    n_total = len(f_itemsets)
    n_1 = len(f_itemsets[f_itemsets['length'] == 1])
    n_2 = len(f_itemsets[f_itemsets['length'] == 2])
    n_3 = len(f_itemsets[f_itemsets['length'] == 3])
    n_4 = len(f_itemsets[f_itemsets['length'] >= 4])
    
    fpgrowth_results.append({
        'Algorithm': 'FP-Growth',
        'Minimum Support': sup,
        'Execution Time (s)': round(elapsed, 2),
        'Total Itemsets': n_total,
        '1-itemsets': n_1,
        '2-itemsets': n_2,
        '3-itemsets': n_3,
        '4+-itemsets': n_4,
        'Status': 'Completed'
    })
    print(f"min_support = {sup:<5}: {elapsed:.2f}s | Total: {n_total} itemsets (1-item: {n_1}, 2-item: {n_2}, 3-item: {n_3}, 4+-item: {n_4})")

In [ ]:
fpgrowth_summary_df = pd.DataFrame(fpgrowth_results)
fpgrowth_summary_df.to_csv('../results/frequent_itemsets/fpgrowth_experiments_summary.csv', index=False)
fpgrowth_summary_df

---
## 5. Comparative Evaluation: Apriori vs. FP-Growth

We load Apriori experiment records and perform a direct head-to-head comparison.

In [ ]:
# Benchmark comparison table
comp_table = pd.read_csv('../results/frequent_itemsets/apriori_vs_fpgrowth_comparison.csv')
comp_table

### Verification of Result Equivalence
We verify whether FP-Growth and Apriori produce identical frequent itemsets at `min_support = 0.01`.

In [ ]:
apriori_01 = pd.read_csv('../results/frequent_itemsets/apriori_itemsets_min_sup_0_01.csv')
fp_01 = fpgrowth_itemsets_dict[0.01]

print(f"Apriori itemset count (min_sup=0.01)   : {len(apriori_01)}")
print(f"FP-Growth itemset count (min_sup=0.01) : {len(fp_01)}")

# Check itemset equivalence
set_apriori = set(apriori_01['itemsets'].apply(lambda x: frozenset(x.split(', '))))
set_fpgrowth = set(fp_01['itemsets'])

match = set_apriori == set_fpgrowth
print(f"Do Apriori and FP-Growth itemsets match 100%? → {match}")

---
## 6. Execution Time Visualization

In [ ]:
plt.figure(figsize=(10, 6))
labels = ['min_sup = 0.05', 'min_sup = 0.02', 'min_sup = 0.01']
apriori_times = [0.09, 1.51, 72.82]
fpgrowth_times = [1.15, 1.84, 4.07]

x = range(len(labels))
width = 0.35

plt.bar([p - width/2 for p in x], apriori_times, width=width, label='Apriori', color='#E53935')
plt.bar([p + width/2 for p in x], fpgrowth_times, width=width, label='FP-Growth', color='#43A047')

plt.xlabel('Minimum Support Threshold', fontsize=12, fontweight='bold')
plt.ylabel('Execution Time in Seconds (log scale)', fontsize=12, fontweight='bold')
plt.yscale('log')
plt.title('Execution Time Comparison: Apriori vs FP-Growth', fontsize=14, fontweight='bold')
plt.xticks(x, labels, fontsize=11)
plt.legend(fontsize=12)

for i in x:
    plt.text(i - width/2, apriori_times[i] * 1.2, f'{apriori_times[i]}s', ha='center', fontsize=10, fontweight='bold')
    plt.text(i + width/2, fpgrowth_times[i] * 1.2, f'{fpgrowth_times[i]}s', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/graphs/apriori_vs_fpgrowth_execution_time.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to: ../results/graphs/apriori_vs_fpgrowth_execution_time.png")

---
## 7. Deep Dive: FP-Growth Frequent Patterns at `min_support = 0.005`

Because FP-Growth easily handles `min_support = 0.005` (which crashed Apriori), we can explore deeper, multi-item relationships that were previously inaccessible.

In [ ]:
fp_005 = fpgrowth_itemsets_dict[0.005].sort_values(by='support', ascending=False).reset_index(drop=True)

print(f"Total itemsets discovered at min_sup = 0.005: {len(fp_005):,}")
print(f"  1-itemsets : {len(fp_005[fp_005['length'] == 1]):,}")
print(f"  2-itemsets : {len(fp_005[fp_005['length'] == 2]):,}")
print(f"  3-itemsets : {len(fp_005[fp_005['length'] == 3]):,}")
print(f"  4-itemsets : {len(fp_005[fp_005['length'] == 4]):,}")

In [ ]:
# Display 4-item frequent patterns
itemsets_4 = fp_005[fp_005['length'] >= 4].copy()
itemsets_4['itemsets'] = itemsets_4['itemsets'].apply(lambda x: ' + '.join(list(x)))
itemsets_4[['itemsets', 'support']]

---
## 8. Save Frequent Itemsets to CSV

In [ ]:
# Save 0.01 and 0.005 FP-Growth itemsets
for sup in [0.01, 0.005]:
    df_to_save = fpgrowth_itemsets_dict[sup].copy()
    df_to_save['itemsets'] = df_to_save['itemsets'].apply(lambda x: ', '.join(list(x)))
    sup_str = str(sup).replace('.', '_')
    path = f'../results/frequent_itemsets/fpgrowth_itemsets_min_sup_{sup_str}.csv'
    df_to_save.to_csv(path, index=False)
    print(f"Saved: {path}")

## 9. Key Observations & Findings

1. **Algorithmic Correctness**: Both Apriori and FP-Growth produced **100% identical frequent itemsets** for all comparable support levels (712 itemsets at `min_support = 0.01`).
2. **Dramatic Speedup**: At `min_support = 0.01`, FP-Growth executed in **4.07 seconds** compared to Apriori's **72.82 seconds** — an **17.9× speedup**.
3. **Memory Efficiency**: At `min_support = 0.005`, Apriori crashed due to a **59.1 GiB candidate generation MemoryError**, while FP-Growth completed smoothly in **5.62 seconds**, identifying 2,175 itemsets including 4-item product bundles.
4. **Why FP-Growth Wins**:
   - **No Candidate Generation**: Avoids calculating Cartesian products ($\\binom{n}{k}$). 
   - **Only 2 Database Scans**: Pass 1 for item frequencies, Pass 2 to construct the compact tree.
   - **Prefix Node Sharing**: Repeated patterns compress into single paths in the FP-Tree.

---
*Next step → `06_association_rules.ipynb` — Generate and evaluate association rules (Support, Confidence, Lift, Leverage, Conviction).*